# beniget: 5 Basic Usage Examples

In [ ]:
# Example 1: Basic beniget import and usage
import beniget

# Check if beniget is properly imported
print(f"beniget module: {beniget.__name__}")
print(f"beniget version: {getattr(beniget, '__version__', 'unknown')}")

# Check available classes
print(f"Available classes: {[attr for attr in dir(beniget) if not attr.startswith('_')]}")

# Test basic class creation
try:
    duc = beniget.DefUseChains()
    print(f"DefUseChains created: {type(duc).__name__}")
    print("Beniget basic functionality works")
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"beniget example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Beniget import and basic test completed")

In [ ]:
# Example 2: Def-use and use-def chains for a small module
import ast

import beniget

module = ast.parse("x = 1\nprint(x)")
duc = beniget.DefUseChains()
duc.visit(module)

# locals maps each scope node to the definitions made in it
(x_def,) = [d for d in duc.locals[module] if d.name() == "x"]
users = x_def.users()
assert len(users) == 1, users  # the assignment to x is read exactly once, by print(x)
(use,) = users
assert isinstance(use.node, ast.Name) and use.node.id == "x" and isinstance(use.node.ctx, ast.Load)
print(f"x is defined on line {x_def.node.lineno} and used on line {use.node.lineno}")

# UseDefChains inverts the mapping: each use points back at the definitions that reach it
udc = beniget.UseDefChains(duc)
assert [d.node for d in udc.chains[use.node]] == [x_def.node]
print(f"use-def chains cover {len(udc.chains)} nodes")

In [ ]:
# Example 3: Report names that are used but never defined
import ast

import beniget


class RecordingChains(beniget.DefUseChains):
    # DefUseChains calls unbound_identifier() for each name it cannot resolve; the default warns
    def __init__(self):
        super().__init__()
        self.unbound = []

    def unbound_identifier(self, name, node):
        self.unbound.append((name, node.lineno))


chains = RecordingChains()
chains.visit(ast.parse("def f():\n    return undefined_name\nprint(len([]))"))
print(f"unbound identifiers: {chains.unbound}")
# Builtins such as print and len resolve; only the undefined name is reported
assert chains.unbound == [("undefined_name", 2)], chains.unbound

In [ ]:
# Example 4: Ancestors records the chain of parent nodes
import ast

import beniget

module = ast.parse("x = 1\nprint(x)")
ancestors = beniget.Ancestors()
ancestors.visit(module)

call = module.body[1].value
x_use = call.args[0]
assert ancestors.parent(call) is module.body[1]
parent_types = [type(node).__name__ for node in ancestors.parents(x_use)]
assert parent_types == ["Module", "Expr", "Call"], parent_types
print(f"parents of x in print(x): {parent_types}")

In [ ]:
# Example 5: Test all beniget classes together
import beniget

# Test all main classes
print("Testing all beniget classes:")

try:
    # Test DefUseChains
    duc = beniget.DefUseChains()
    print(f"✓ DefUseChains: {type(duc).__name__}")
    
    # Test UseDefChains
    udc = beniget.UseDefChains(duc)
    print(f"✓ UseDefChains: {type(udc).__name__}")
    
    # Test Ancestors
    ancestors = beniget.Ancestors()
    print(f"✓ Ancestors: {type(ancestors).__name__}")
    
    print("All beniget classes created successfully")
    
except Exception as e:
    print(f"Error in comprehensive test: {e}")
    raise AssertionError(
        f"beniget example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Beniget comprehensive test completed")